In [41]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.ndimage import gaussian_filter1d
%matplotlib inline
from ipywidgets import interact, fixed
import ipywidgets as widgets
from numba import njit, prange
from numba import float64, int32
from numba.experimental import jitclass
import pickle
from typing import List, Tuple, Callable, Any

from utils import MovingFunctionCalculator
from utils import Analytical
from utils import Plotter

In [46]:
class Simulation:
    def __init__(self, L, T, kon, koff, kstep, kq, q):
        self.L:int = L
        self.T:float = T
        self.kon:float = kon
        self.koff:float= koff
        self.kstep:float= kstep
        self.kq:float= kq
        self.q:float= q
        self.grid:np.ndarray[Any, np.dtype[np.float64]] = np.zeros(L)
        self.grid_activated:np.ndarray[Any, np.dtype[np.float64]] = np.zeros(L)

    @njit
    def activation_to_factors(self)->np.ndarray[Any, np.dtype[np.float64]]:
        return np.ones(10) #+ (self.q-1) * self.grid_activated

    @njit
    def calc_propensities(self):        
        aon = self.kon * np.sum( self.activation_to_factors())
        aoff = self.koff * np.sum(self.grid)
        astep = self.kstep * np.sum(self.grid)
        aq = self.kq * len(np.where((self.grid_activated==1)&(self.grid==0))[0])

        return np.array([aon, aoff, astep, aq])

    #---------------------------------------------------------------------------

    @njit
    def bind_kinesin(self) -> None:
        S = np.cumsum(self.activation_to_factors())
        r = S[-1]*np.random.random()
        side = np.argmax(S>=r)
        self.grid[side] += 1

    @njit
    def unbind_kinesin(self) -> None:
        S = np.cumsum(self.grid)
        r = np.random.randint(low = 1, high = int(S[-1])+1)
        side = np.argmax(S>=r)

        self.grid[side] -= 1

    @njit
    def move_kinesin_w_fall(self) -> None:
        S = np.cumsum(self.grid)
        r = np.random.randint(low = 1, high = int(S[-1])+1)
        side = np.argmax(S>=r)

        self.grid[side] -= 1
        if side != len(self.grid)-1:
            self.grid[side+1] += 1
    
    @njit
    def deactivate_tubule(self) -> None:
        sides = np.where((self.grid_activated==1)&(self.grid==0))[0]
        S = len(sides)
        r = np.random.randint(S)
        side = sides[r]
        self.grid_activated[side]=0

    #---------------------------------------------------------------------------
    @njit
    def step(self)->Tuple[float, int]:
        r1 = np.random.uniform()
        r2 = np.random.uniform()

        A = self.calc_propensities()
        R_tot = np.sum(A)
        
        A_normalised = np.cumsum(A)/R_tot

        dt = (1/R_tot) * np.log(1/r1)
        idx = np.argwhere(A_normalised>r2)[0][0]
        return dt, int(idx)
    
    @njit
    def simulation(self):
        time = 0.0
        block = 0
        period = 0.1
        next_write_time = period
        blocks = int(self.T/period)


        DATA = np.zeros((blocks, self.L))
        TIMES= np.zeros(blocks)

        while time<self.T:
            dt, idx = self.step()
            time+= dt      

            # Action
            if idx==0:
                self.bind_kinesin()
            elif idx==1:
                self.unbind_kinesin()
            elif idx==2:
                self.move_kinesin_w_fall()
            else:
                self.deactivate_tubule()
            
            
            #Write    
            if next_write_time<time:
                DATA[block,:] = self.grid
                TIMES[block] = time
                block+=1
                next_write_time+=period

        return DATA[:block], TIMES[:block]


TypeError: class members are not yet supported: activation_to_factors, calc_propensities, bind_kinesin, unbind_kinesin, move_kinesin_w_fall, deactivate_tubule, step, simulation

In [47]:
L = 30
T = 1
kon = 0.8
koff = 1
kstep = 10
kq = 5
q = 10


In [48]:
sim = Simulation( L, T, kon, koff, kstep, kq, q)

In [49]:
sim.activation_to_factors()

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type pyobject
During: typing of argument at /var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/2967896968.py (13)

File "../../../../var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/2967896968.py", line 13:
<source missing, REPL/exec in use?> 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class '__main__.Simulation'> 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class '__main__.Simulation'> 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class '__main__.Simulation'>


In [28]:
sim.calc_propensities()

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type pyobject
During: typing of argument at /var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/3931805251.py (17)

File "../../../../var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/3931805251.py", line 17:
<source missing, REPL/exec in use?> 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class '__main__.Simulation'>


In [14]:
data, time = sim.simulation()

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
non-precise type pyobject
During: typing of argument at /var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/1072043859.py (76)

File "../../../../var/folders/89/w_60r_4x0cj3g1zv6zj2235h0000gn/T/ipykernel_26920/1072043859.py", line 76:
<source missing, REPL/exec in use?> 

This error may have been caused by the following argument(s):
- argument 0: Cannot determine Numba type of <class '__main__.Simulation'>


In [7]:
a = np.array([0,0,1,1,1,0,0,0,1,1,0])
b = np.array([0,2,3,1,0,2,3,0,0,2,0])


In [10]:
len(np.where((a==1)&(b==0))[0])

2